In [ ]:
# ScoutAI - Auditoría del Dataset

## Objetivo
# Analizar los archivos CSV recibidos para comprobar su calidad antes de realizar la limpieza con Python y Pandas.

## Comprobaciones
# - Valores nulos
# - Registros duplicados
# - Tipos de datos
# - Valores incoherentes
# - Identificadores y relaciones entre tablas

# Importante: los archivos originales nunca se modificarán.

#trabajando en la primera fase de auditoría de datos de OFFBALL.

In [ ]:
#importo las librerías necesarias

import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# lo conectamos con drive
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# compruebamos carpetas

from pathlib import Path

OFFBALL_DIR = Path(
    "/content/drive/MyDrive/PROYECTO-IGOR-CARMELO-BYRON/OFFBALL"
)

RAW_DIR = OFFBALL_DIR / "csv_originales"

print("¿Existe la carpeta OFFBALL?", OFFBALL_DIR.exists())
print("¿Existe csv_originales?", RAW_DIR.exists())
print("¿Existe Machine_Learning?", (OFFBALL_DIR / "Machine_Learning").exists())

In [ ]:
## compruebo que Colab encuentra los cinco CSV
csv_files = sorted(RAW_DIR.rglob("*.csv"))

print("Número de archivos CSV:", len(csv_files))

for file in csv_files:
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{file.name} — {size_mb:.2f} MB")

In [ ]:
#inspecciono las columnas, los tipos de datos y algunas filas de cada archivo

import pandas as pd
from IPython.display import display

for file in csv_files:
    print("\n" + "=" * 70)
    print("ARCHIVO:", file.name)

    # Leer solamente 3 filas para inspeccionar
    df_preview = pd.read_csv(file, nrows=3)

    print("\nNúmero de columnas:", len(df_preview.columns))
    print("\nNombres de las columnas:")
    print(df_preview.columns.tolist())

    print("\nTipos de datos detectados:")
    print(df_preview.dtypes)

    print("\nPrimeras 3 filas:")
    display(df_preview)

    del df_preview

In [ ]:
#comprobado todos los CSV para ver qué datos están vacíos y cuántos faltan en cada columna

def auditar_nulos_csv(ruta, chunksize=100_000):

    total_filas = 0
    nulos_totales = None
    tipos = None

    # Leer el archivo por bloques para no cargarlo entero
    for bloque in pd.read_csv(
        ruta,
        chunksize=chunksize,
        low_memory=False
    ):

        total_filas += len(bloque)

        nulos_bloque = bloque.isna().sum()

        if nulos_totales is None:
            nulos_totales = nulos_bloque
            tipos = bloque.dtypes.astype(str)
        else:
            nulos_totales = nulos_totales.add(
                nulos_bloque,
                fill_value=0
            )

    resumen = pd.DataFrame({
        "tipo_detectado": tipos,
        "nulos": nulos_totales.astype(int)
    })

    resumen["porcentaje_nulos"] = (
        resumen["nulos"] / total_filas * 100
    ).round(2)

    return total_filas, resumen.sort_values(
        "porcentaje_nulos",
        ascending=False
    )


# Analizar los cinco CSV
for file in csv_files:

    print("\n" + "=" * 70)
    print("AUDITORÍA:", file.name)

    filas, resultado = auditar_nulos_csv(file)

    print("Total de filas:", filas)
    print("Total de columnas:", len(resultado))
    print("Columnas con valores nulos:",
          (resultado["nulos"] > 0).sum())

    display(resultado)

In [ ]:
# Todavía no he cambiado ningún archivo, primero estoy apuntando los problemas que he encontrado para decidir cómo solucionarlos después.

#siguientes pasos:

#Comprobar duplicados en los cinco CSV.
#Revisar las fechas y comprobar que tienen un formato correcto.
#Revisar los datos incoherentes.
#Preparar la limpieza con Pandas,



In [ ]:

#revisar datos duplicados
claves = {
    "appearances.csv": ["appearance_id"],
    "clubs.csv": ["club_id"],
    "games.csv": ["game_id"],
    "player_valuations.csv": ["player_id", "date"],
    "players.csv": ["player_id"]
}

for nombre, columnas in claves.items():
    ruta = RAW_DIR / nombre

    print("\n" + "=" * 60)
    print("ARCHIVO:", nombre)

    df = pd.read_csv(ruta, usecols=columnas, dtype="string")

    repetidos = df.duplicated(subset=columnas, keep=False)

    print("Filas analizadas:", len(df))
    print("Filas con posibles duplicados:", repetidos.sum())

    if repetidos.sum() > 0:
        display(df.loc[repetidos].sort_values(columnas).head(10))

    del df

In [ ]:
#Vamos a comprobar que las fechas de los archivos se pueden interpretar correctamente y detectar posibles fechas incorrectas.

fechas = {
    "appearances.csv": ["date"],
    "games.csv": ["date"],
    "player_valuations.csv": ["date"],
    "players.csv": ["date_of_birth", "contract_expiration_date"]
}

for nombre, columnas in fechas.items():
    print("\n" + "=" * 60)
    print("ARCHIVO:", nombre)

    for columna in columnas:
        df = pd.read_csv(
            RAW_DIR / nombre,
            usecols=[columna],
            dtype="string"
        )

        valores = df[columna].str.strip()
        vacios = valores.isna() | valores.eq("")

        convertidas = pd.to_datetime(
            valores.mask(vacios),
            errors="coerce"
        )

        invalidas = (~vacios) & convertidas.isna()

        print("\nColumna:", columna)
        print("Fechas vacías:", vacios.sum())
        print("Fechas no válidas:", invalidas.sum())

        if convertidas.notna().any():
            print("Fecha más antigua:", convertidas.min())
            print("Fecha más reciente:", convertidas.max())

        del df

In [ ]:
#Ahora vamos a comprobar que los identificadores coinciden entre tablas
# Cargar los identificadores de referencia

ids_ref = {
    "players": set(
        pd.read_csv(RAW_DIR / "players.csv",
                    usecols=["player_id"], dtype="string")["player_id"].dropna()
    ),
    "games": set(
        pd.read_csv(RAW_DIR / "games.csv",
                    usecols=["game_id"], dtype="string")["game_id"].dropna()
    ),
    "clubs": set(
        pd.read_csv(RAW_DIR / "clubs.csv",
                    usecols=["club_id"], dtype="string")["club_id"].dropna()
    )
}

# Relaciones que vamos a comprobar
relaciones = [
    ("appearances.csv", "player_id", "players"),
    ("appearances.csv", "game_id", "games"),
    ("appearances.csv", "player_club_id", "clubs"),
    ("appearances.csv", "player_current_club_id", "clubs"),
    ("games.csv", "home_club_id", "clubs"),
    ("games.csv", "away_club_id", "clubs"),
    ("player_valuations.csv", "player_id", "players"),
    ("player_valuations.csv", "current_club_id", "clubs"),
    ("players.csv", "current_club_id", "clubs")
]

# Comprobar las relaciones
for archivo, columna, tabla_ref in relaciones:
    df = pd.read_csv(
        RAW_DIR / archivo,
        usecols=[columna],
        dtype="string"
    )

    valores = df[columna].str.strip()
    validos = valores.notna() & valores.ne("")
    no_encontrados = validos & ~valores.isin(ids_ref[tabla_ref])

    print("\n" + "=" * 55)
    print(f"{archivo} -> {tabla_ref}")
    print("Referencias comprobadas:", validos.sum())
    print("Sin coincidencia:", no_encontrados.sum())

    if no_encontrados.sum() > 0:
        print("Ejemplos:", valores[no_encontrados].drop_duplicates().head(5).tolist())

    del df

In [ ]:
# investigar los identificadores que faltan
for archivo, columna, tabla_ref in relaciones:
    df = pd.read_csv(
        RAW_DIR / archivo,
        usecols=[columna],
        dtype="string"
    )

    valores = df[columna].str.strip()
    faltantes = valores[
        valores.notna()
        & valores.ne("")
        & ~valores.isin(ids_ref[tabla_ref])
    ]

    print("\n" + "=" * 55)
    print(archivo, "->", tabla_ref)
    print("Filas sin coincidencia:", len(faltantes))
    print("Identificadores diferentes:", faltantes.nunique())

    print("\nIdentificadores que más se repiten:")
    display(faltantes.value_counts().head(10))

    del df

In [ ]:
#investigar los clubes que faltan
# Obtener los clubes disponibles
clubes = pd.read_csv(
    RAW_DIR / "clubs.csv",
    usecols=["club_id", "name"],
    dtype="string"
)

ids_clubes = set(clubes["club_id"].str.strip().dropna())

print("Clubes registrados en clubs.csv:", len(ids_clubes))


def revisar_clubes(archivo, id_col, nombre_col):

    df = pd.read_csv(
        RAW_DIR / archivo,
        usecols=[id_col, nombre_col],
        dtype="string"
    )

    ids = df[id_col].str.strip()
    nombres = df[nombre_col].str.strip()

    mask = (
        ids.notna()
        & ids.ne("")
        & ~ids.isin(ids_clubes)
    )

    faltantes = pd.DataFrame({
        "club_id": ids[mask],
        "nombre": nombres[mask]
    })

    print("\n", "=" * 55)
    print(archivo, "-", id_col)
    print("Identificadores sin coincidencia:", faltantes["club_id"].nunique())

    resumen = (
        faltantes.groupby(["club_id", "nombre"], dropna=False)
        .size()
        .reset_index(name="filas")
        .sort_values("filas", ascending=False)
        .head(15)
    )

    display(resumen)

    del df


revisar_clubes("games.csv", "home_club_id", "home_club_name")
revisar_clubes("games.csv", "away_club_id", "away_club_name")
revisar_clubes("player_valuations.csv", "current_club_id", "current_club_name")
revisar_clubes("players.csv", "current_club_id", "current_club_name")

In [ ]:
#investigar el jugador que falta

df = pd.read_csv(
    RAW_DIR / "appearances.csv",
    usecols=["player_id", "player_name", "date", "game_id"],
    dtype="string"
)

display(df[df["player_id"].str.strip() == "380365"])

In [ ]:
# Buscar el jugador en la tabla principal de jugadores

players = pd.read_csv(
    RAW_DIR / "players.csv",
    dtype={"player_id": "string"}
)

jugador = players[
    players["player_id"].str.strip() == "380365"
]

print("Número de coincidencias:", len(jugador))

display(jugador)